<a href="https://colab.research.google.com/github/Haross/sql_notebook/blob/main/SPARK_SQL_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set up

In [1]:
from pyspark.sql import SparkSession
import json

if not 'spark' in globals():
  spark = SparkSession.builder.getOrCreate()

We will import the files needed for the exercises as follows:

In [20]:
%%capture
!wget -O "country.json" "https://raw.githubusercontent.com/Haross/DB_pics_nt/main/country.json"
!wget -O "laureate.json" "https://raw.githubusercontent.com/Haross/DB_pics_nt/main/laureate.json"
!wget -O "prize.json" "https://raw.githubusercontent.com/Haross/DB_pics_nt/main/prize.json"
!wget -O "country_clean.json" "https://raw.githubusercontent.com/Haross/DB_pics_nt/main/country_clean.json"


# Warm up


Before doing the exercises, we present a couple of examples that should serve you as reference on how to proceed with the exercises. Precisely, we will process data related
to the Nobel prizes as obtained from their [API](https://nobelprize.readme.io/docs/getting-started). Data provided by such API has JSON format, please make sure to get familiar with the format if you are not familiar with it.


## Example 1

The goal of this example is to show you how to, using PySpark, execute SQL queries over unstructured data.

We will proceed to read the file and print the inferred schema of `country.json`

In [ ]:
fileDF = spark.read.json("/content/country.json")
fileDF.printSchema()
# We will create a temporal view for the DataFrame `fileDF`
fileDF.createOrReplaceTempView("countriesArray")

root
 |-- countries: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- code: string (nullable = true)
 |    |    |-- name: string (nullable = true)



In [ ]:
fileDF.show()

+--------------------+
|           countries|
+--------------------+
|[{DE, Alsace, the...|
+--------------------+



In [ ]:
#The LATERAL VIEW clause will create a new row for each element in the array "countries".
#Precisely, it will generate a struct called countryInfo (see the generated schema printed in line 13)
countriesDF = spark.sql("SELECT countryInfo FROM countriesArray LATERAL VIEW explode(countries) AS countryInfo")
countriesDF.printSchema()

root
 |-- countryInfo: struct (nullable = true)
 |    |-- code: string (nullable = true)
 |    |-- name: string (nullable = true)



In [ ]:
countriesDF.show()

+--------------------+
|         countryInfo|
+--------------------+
|{DE, Alsace, then...|
|        {DE, Alsace}|
|       {DE, Germany}|
|     {AR, Argentina}|
|     {AU, Australia}|
|       {AT, Austria}|
|       {BE, Belgium}|
|         {MM, Burma}|
|       {MM, Myanmar}|
|        {CA, Canada}|
|         {CL, Chile}|
|         {CN, China}|
|      {CO, Colombia}|
|    {CR, Costa Rica}|
|{CZ, Czechoslovakia}|
|{VN, Democratic R...|
|       {DK, Denmark}|
|    {TL, East Timor}|
|         {EG, Egypt}|
|{DE, Federal Repu...|
+--------------------+
only showing top 20 rows



Now, we can query back this dataframe.

In [ ]:
countriesDF.createOrReplaceTempView("countries")
#The function collect_list is an aggregation function that returns a list for an attribute in the group by set
res = spark.sql("SELECT countryInfo.code, collect_list(countryInfo.name) FROM countries GROUP BY countryInfo.code")
res.show()

+----+------------------------------+
|code|collect_list(countryInfo.name)|
+----+------------------------------+
|  MM|          [Burma, Myanmar, ...|
|  LT|          [Lithuania, Russi...|
|  DZ|                     [Algeria]|
|  FI|          [Finland, Russian...|
|  AZ|          [Russian Empire (...|
|  UA|          [Austria-Hungary ...|
|  RO|                     [Romania]|
|  ZM|          [Northern Rhodesi...|
|  NL|             [the Netherlands]|
|  PL|          [Poland, Poland, ...|
|  MK|          [Ottoman Empire (...|
|  MX|                      [Mexico]|
|  CN|                [China, Tibet]|
|  AT|          [Austria, Austria...|
|  RU|          [Russia, USSR, Pr...|
|  HR|          [Austria-Hungary ...|
|  CZ|          [Czechoslovakia, ...|
|  PT|                    [Portugal]|
|  GH|          [Ghana, Gold Coas...|
|  LR|                     [Liberia]|
+----+------------------------------+
only showing top 20 rows



# Exercise 1

Using Spark, return for each year, the total number of laureates that got a Nobel prize. An excerpt of your output should look like the following (not necessarily ordered):
* 2017 -- 12
* 2016 -- 11
* 2015 – 11

In [5]:
fileDF = spark.read.json('/content/laureate.json')
fileDF.printSchema()

root
 |-- laureates: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- born: string (nullable = true)
 |    |    |-- bornCity: string (nullable = true)
 |    |    |-- bornCountry: string (nullable = true)
 |    |    |-- bornCountryCode: string (nullable = true)
 |    |    |-- died: string (nullable = true)
 |    |    |-- diedCity: string (nullable = true)
 |    |    |-- diedCountry: string (nullable = true)
 |    |    |-- diedCountryCode: string (nullable = true)
 |    |    |-- firstname: string (nullable = true)
 |    |    |-- gender: string (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- prizes: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- affiliations: array (nullable = true)
 |    |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |    |-- category: string (nullable = true)
 |    |    |    |    |-- motivation: string (nullable = 

In [ ]:

fileDF = spark.read.json('/content/prize.json')
fileDF.createOrReplaceTempView("prizesArray")
res = spark.sql("""

""")
res.show()

+----+--------+
|year|count(e)|
+----+--------+
|2017|      12|
|2016|      11|
|2015|      11|
|2014|      13|
|2013|      13|
|2012|      10|
|2011|      13|
|2010|      11|
|2009|      13|
|2008|      12|
|2007|      12|
|2006|       9|
|2005|      13|
|2004|      12|
|2003|      11|
|2002|      13|
|2001|      15|
|2000|      13|
|1999|       7|
|1998|      12|
+----+--------+
only showing top 20 rows



# Exercise 2

Using Spark, return the name and date of birth of those laureates that got a Nobel prize motivated by their work on DNA. An excerpt of your output should look like the following:
* Berg, Paul -- 1926-06-30
* Lindahl, Tomas -- 1938-01-28
* Modrich, Paul -- 1946-06-13
* Sancar, Aziz -- 1946-09-08

In [ ]:
# TODO: Replace <FILL IN> with appropriate code
fileDF = spark.read.json("/content/laureate.json")
fileDF.createOrReplaceTempView("laureatesArray")
res = spark.sql("""

""")
res.show()

+---------+-------+----------+
|firstname|surname|      born|
+---------+-------+----------+
|     Paul|   Berg|1926-06-30|
|    Tomas|Lindahl|1938-01-28|
|     Paul|Modrich|1946-06-13|
|     Aziz| Sancar|1946-09-08|
+---------+-------+----------+



# Exercise 3

Using Spark, return for each Nobel prize category the total number of prizes awarded.

An excerpt of the output should look like:
* physics -- 111
* literature -- 110
* chemistry -- 109

In [6]:
fileDF = spark.read.json('/content/prize.json')
fileDF.createOrReplaceTempView("prizesArray")

res = spark.sql("""

""")

res.show()

+----------+------------+
|  category|total_prizes|
+----------+------------+
|   physics|         111|
|literature|         110|
| chemistry|         109|
|  medicine|         108|
|     peace|          98|
| economics|          49|
+----------+------------+



# Exercise 4

Using Spark, return the countries of birth that have produced at least 10 laureates, together with the number of laureates born there.

An excerpt of the output should look like:

* USA - 263
* United Kingdom - 83
* Germany - 63

In [8]:
fileDF = spark.read.json("/content/laureate.json")
fileDF.createOrReplaceTempView("laureatesArray")

res = spark.sql("""


""")

res.show()

+---------------+---------------+
|    bornCountry|total_laureates|
+---------------+---------------+
|            USA|            263|
| United Kingdom|             83|
|        Germany|             63|
|         France|             51|
|         Sweden|             29|
|          Japan|             25|
|the Netherlands|             18|
|         Canada|             18|
|         Russia|             17|
|          Italy|             17|
|    Switzerland|             17|
|        Austria|             14|
|         Norway|             12|
|          China|             11|
|        Denmark|             11|
|       Scotland|             11|
|      Australia|             10|
+---------------+---------------+



# Exercise 5

In this exercise, we will join laureates with their country of birth.

The original file `country.json` contains multiple names for the same country code
(e.g., historical regions like "Prussia (now Germany)" or "Alsace").
This would produce duplicate or misleading results when joining.

To simplify the task, we will use a cleaned version of the dataset,
`country_clean.json`, which contains exactly one country name per code.

Using Spark:

- Load `country_clean.json` and `laureate.json`
- Flatten both datasets
- Join them using the country code
- Return the name of each laureate together with the name of their country of birth

Show the first 20 results.

An excerpt of the output should look like:

- Wilhelm Conrad Röntgen - Germany              
- Hendrik Antoon Lorentz - the Netherlands       
- Pieter Zeeman - the Netherlands     
- Antoine Henri Becquerel - France                
- Pierre Curie - France               
- Marie Curie, née Sklodowska - Poland                

+-----------------------------------+----------------------+
|name                               |country               |
+-----------------------------------+----------------------+
|Wilhelm Conrad Röntgen             |Germany               |
|Hendrik Antoon Lorentz             |the Netherlands       |
|Pieter Zeeman                      |the Netherlands       |
|Antoine Henri Becquerel            |France                |
|Pierre Curie                       |France                |
|Marie Curie, née Sklodowska        |Poland                |
|Lord Rayleigh (John William Strutt)|United Kingdom        |
|Philipp Eduard Anton von Lenard    |Hungary (now Slovakia)|
|Joseph John Thomson                |United Kingdom        |
|Albert Abraham Michelson           |Poland                |
|Gabriel Lippmann                   |Luxembourg            |
|Guglielmo Marconi                  |Italy                 |
|Karl Ferdinand Braun               |Germany               |
|Johannes Diderik van de

# Exercise 6

Find the year or years with the maximum number of laureates receiving a Nobel prize.

In [17]:
# This is an optional template to solve exercise 6, you can solve this exercise as you prefer e.g. one single query.

fileDF = spark.read.json('/content/prize.json')
fileDF.createOrReplaceTempView("prizesArray")

yearCounts = spark.sql("""

""")
yearCounts.createOrReplaceTempView("yearCounts")

res = spark.sql("""
  SELECT _______, _________
  FROM yearCounts
  WHERE ____________ = (SELECT MAX(_________) FROM ________)
""")

res.show()

+----+---------------+
|year|total_laureates|
+----+---------------+
|2001|             15|
+----+---------------+

